# 01. Setup, Data, and how we're scored

**Goal of this notebook:** get the data onto our machine, look at it with our own eyes, and understand *exactly* how the organizers score a submission, before we write a single line of steering code.

**Runs on:** Mac, CPU only. No GPU needed.

**Where this file lives:** `notebooks/01_setup_and_data.ipynb` inside the fork of the organizers' repo. The first cell moves the working directory up to the repo root, so `clas/`, `evaluate.py` and `data/` all resolve.

## Step 1 - Point the notebook at the repo root

The organizers' code is a small package (`clas/`) plus scripts (`download_data.py`, `evaluate.py`). We **reused their code instead of rewriting it**. That way our numbers are computed the same way the leaderboard computes them.

In [9]:
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)
print("has clas/ package:", (ROOT / "clas").exists())

repo root: /Users/karan/Documents/Fall '26/NLP/SemEval Project/multilingual_language_steering
has clas/ package: True


## Step 2 - Check our Hugging Face login

Three things we need are **gated** on Hugging Face: we need to "Agree" on each page once, while logged in:

- `openlanguagedata/flores_plus` (training sentences)
- `meta-llama/Llama-3.1-8B-Instruct` (model #1; approval can take a few hours)
- `CohereLabs/tiny-aya-global` (model #2)

Then run `hf auth login` **in the terminal** (with a *read* token from huggingface.co/settings/tokens). This cell just confirms the login worked.

In [10]:
from huggingface_hub import whoami
print("logged in as:", whoami()["name"])

logged in as: KaranCheemalapati


## Step 3 - Download the data

`download_data.py` builds two folders:

| folder | what | size | what we use it for |
|---|---|---|---|
| `data/train/` | FLORES-200 dev+devtest: the **same ~2000 sentences translated into every language** | ~2009 lines / language | finding *where* language lives inside the model (next notebook) |
| `data/validation/` | CLaS-Bench: **70 open-ended questions, parallel across languages** | 70 lines / language | measuring how well our steering works |

The real test set (100 unseen questions/language) stays hidden.

This downloads 23 languages, so give it a few minutes. It skips files that already exist, so re-running is safe.

In [11]:
!{sys.executable} download_data.py --out data

=  ar train      exists, skipping
=  ar validation exists, skipping
=  cs train      exists, skipping
=  cs validation exists, skipping
=  de train      exists, skipping
=  de validation exists, skipping
=  el train      exists, skipping
=  el validation exists, skipping
=  en train      exists, skipping
=  en validation exists, skipping
=  es train      exists, skipping
=  es validation exists, skipping
=  fa train      exists, skipping
=  fa validation exists, skipping
=  fr train      exists, skipping
=  fr validation exists, skipping
=  hi train      exists, skipping
=  hi validation exists, skipping
=  it train      exists, skipping
=  it validation exists, skipping
=  ja train      exists, skipping
=  ja validation exists, skipping
=  ka train      exists, skipping
=  ka validation exists, skipping
=  ko train      exists, skipping
=  ko validation exists, skipping
=  mt train      exists, skipping
=  mt validation exists, skipping
=  no train      exists, skipping
=  no validati

## Step 4 - Look at the validation questions

Key idea: line *i* in `en.txt` is the **same question** as line *i* in `de.txt`, `ja.txt`, and so on. So for a pair like `ar -> de`, the model gets the Arabic question and should answer in German.

In [12]:
import pandas as pd
from clas import languages as L
from clas.io_utils import load_questions, read_lines

LANGS = L.HIGH + L.LOW
questions = load_questions(Path("data/validation"), LANGS)
print({c: len(q) for c, q in questions.items()})

pd.set_option("display.max_colwidth", 120)
pd.DataFrame({c: questions[c][:3] for c in ["en", "de", "ar", "ja", "hi", "sw"]}).T

{'ar': 70, 'de': 70, 'en': 70, 'es': 70, 'fr': 70, 'ja': 70, 'zh': 70, 'ru': 70, 'hi': 70, 'ko': 70, 'cs': 70, 'el': 70, 'fa': 70, 'vi': 70, 'uk': 70, 'mt': 70, 'no': 70, 'sw': 70, 'th': 70, 'tr': 70}


,0,1,2
en,How can I improve my time management skills?,What are the most effective ways to deal with stress?,What are the main differences between Python and JavaScript programming languages?
de,Wie kann ich meine Zeitmanagementfähigkeiten verbessern?,"Was sind die effektivsten Möglichkeiten, mit Stress umzugehen?",Was sind die Hauptunterschiede zwischen Python- und JavaScript-Programmiersprachen?
ar,كيف يمكنني تحسين مهارات إدارة الوقت؟,ما هي الطرق الأكثر فعالية للتعامل مع التوتر؟,ما هي الاختلافات الرئيسية بين لغات برمجة Python و JavaScript؟
ja,時間管理スキルを向上させるにはどうすればよいですか？,ストレスに対処する最も効果的な方法は何ですか？,PythonとJavaScriptプログラミング言語の主な違いは何ですか？
hi,मैं अपने समय प्रबंधन कौशल को कैसे सुधार सकता हूँ?,तनाव से निपटने के लिए सबसे प्रभावी तरीके क्या हैं?,पायथन और जावास्क्रिप्ट प्रोग्रामिंग भाषाओं के बीच मुख्य अंतर क्या हैं?
sw,Je! Ninawezaje kuboresha ufanisi wangu wa kumudu wakati?,Je! Ni njia gani bora za kukabiliana na msongo wa mawazo?,Je! Kuna tofauti gani kuu kati ya lugha za programu za Python na JavaScript?


**What to notice:** these are open-ended "explain / describe" questions, so answers will be long (up to 1024 tokens). That matters later: steering has to *hold* the target language over a long generation, not just the first few words.

## Step 5 - Look at the FLORES training sentences

Same parallel idea, but these are short news/wiki sentences. Next notebook we'll push all ~2000 sentences of each language through the model, record its hidden states, and average them. The difference between, say, the "German average" and the "all-languages average" is our **steering vector** for German.

Because the sentences say the *same thing* in every language, the content cancels out in that difference and what's left is mostly **language identity**. That's why parallel data is so useful here.

In [13]:
train = {c: read_lines(Path(f"data/train/{c}.txt")) for c in LANGS}
print({c: len(s) for c, s in train.items()})

for c in ["en", "de", "ru", "zh", "th", "mt"]:
    print(f"{c}: {train[c][0]}")

{'ar': 2009, 'de': 2009, 'en': 2009, 'es': 2009, 'fr': 2009, 'ja': 2009, 'zh': 2009, 'ru': 2009, 'hi': 2009, 'ko': 2009, 'cs': 2009, 'el': 2009, 'fa': 2009, 'vi': 2009, 'uk': 2009, 'mt': 2009, 'no': 2009, 'sw': 2009, 'th': 2009, 'tr': 2009}
en: On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.
de: Am Montag haben die Wisenschaftler der Stanford University School of Medicine die Erfindung eines neuen Diagnosetools bekanntgegeben, mit dem Zellen nach ihrem Typ sortiert werden können: ein winziger, ausdruckbarer Chip, der für jeweils etwa einen US-Cent mit Standard-Tintenstrahldruckern hergestellt werden kann.
ru: В понедельник ученые из Медицинской школы Стэнфордского университета объявили об изобретении нового диагностического инструмента, который может сортировать клетки по 

## Step 6 - Tracks -> language pairs -> workload

Let's count how many generations each track needs on the validation set. This tells us how much GPU time to budget.

In [19]:
for track in ["t1", "t2"]:
    p = L.pairs(track)
    print(f"{track.upper()}: {len(p)} pairs x 70 questions = {len(p) * 70:,} generations per model")
print("first few T1 pairs:", L.pairs("t1")[:5])
print("T2 targets:", L.LOW)

T1: 90 pairs x 70 questions = 6,300 generations per model
T2: 100 pairs x 70 questions = 7,000 generations per model
first few T1 pairs: [('ar', 'de'), ('ar', 'en'), ('ar', 'es'), ('ar', 'fr'), ('ar', 'ja')]
T2 targets: ['cs', 'el', 'fa', 'vi', 'uk', 'mt', 'no', 'sw', 'th', 'tr']


**Takeaway:** T1 is 6,300 generations per model on the full validation set. That's too many for every experiment. While tuning we use a **subset** (e.g. 10 questions x ~12 pairs) and only run the full set for final numbers.

## Step 7 - How "language compliance" (LFS) is measured

**LFS = Language Forcing Success**: the fraction of answers that a fastText language-ID model labels as the target language, **with ≥ 0.9 confidence**. For zh/ja/ko there's a fallback that just looks at the script (Han / Kana / Hangul).

In [15]:
from evaluate import load_lid, predict_lang, harmonic
lid = load_lid()
mapping = L.flores_to_code()

tests = {
    "clean German": "Die Photosynthese ist der Prozess, bei dem Pflanzen Licht in chemische Energie umwandeln.",
    "German + English mix": "Die Photosynthese ist ein process where plants convert light into energy, sehr wichtig.",
    "Norwegian": "Fotosyntese er prosessen der planter omdanner lys til kjemisk energi.",
    "short Chinese": "光合作用很重要。",
    "Japanese": "光合成とは、植物が光を化学エネルギーに変えるプロセスです。",
    "gibberish": "asdf qwer zxcv uiop",
}
for name, text in tests.items():
    labels, probs = lid.predict(" ".join(text.split()), k=2)
    top = [(l.replace("__label__", ""), round(float(p), 3)) for l, p in zip(labels, probs)]
    print(f"{name:22s} -> scored as '{predict_lang(lid, text, mapping)}'   raw top-2: {top}")

/opt/anaconda3/envs/clas/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


clean German           -> scored as 'de'   raw top-2: [('deu_Latn', 1.0), ('nld_Latn', 0.0)]
German + English mix   -> scored as 'unk'   raw top-2: [('deu_Latn', 0.658), ('eng_Latn', 0.325)]
Norwegian              -> scored as 'unk'   raw top-2: [('nob_Latn', 0.858), ('nno_Latn', 0.124)]
short Chinese          -> scored as 'zh'   raw top-2: [('bod_Tibt', 0.343), ('eng_Latn', 0.262)]
Japanese               -> scored as 'ja'   raw top-2: [('jpn_Jpan', 0.243), ('zho_Hans', 0.216)]
gibberish              -> scored as 'unk'   raw top-2: [('nld_Latn', 0.865), ('ltz_Latn', 0.029)]


**What to notice:**
- Mixed-language text often falls below the 0.9 confidence bar and counts as a **fail** (`unk`). Half-steered answers get no partial credit.
- Close languages (Norwegian vs Danish/Swedish) can get confused, which is worth remembering for T2.
- The script fallback rescues short CJK text.

## Step 8 - The final score: LSS = harmonic mean of LFS, OR and LQ

- **LFS** (above): right language? In [0, 1].
- **OR = Output Relevance**: an LLM judge (Qwen3-8B) rates each answer 0/1/2 for "relevant, coherent, useful", normalized to [0, 1]. It is told to **ignore** the language.
- **LQ = Language Quality**: the same judge rates 0/1/2 for grammar, spelling and fluency *as a native speaker would*, ignoring whether it answers the question.
- **LSS = harmonic mean of the three**: 3 / (1/LFS + 1/OR + 1/LQ). This is the main metric.

Harmonic mean punishes imbalance. Push steering so hard the model outputs broken German and LFS stays high, but LQ (and OR) crash, and so does LSS.

In [20]:
cases = {"no steering (wrong language, good answer)":    (0.0, 0.95, 0.95),
         "over-steered (right language, gibberish)":     (1.0, 0.05, 0.05),
         "right language, relevant, but broken grammar": (0.95, 0.85, 0.30),
         "balanced":                                     (0.8, 0.8, 0.8)}
for name, (lfs, orl, lq) in cases.items():
    print(f"{name:46s} LFS={lfs:.2f} OR={orl:.2f} LQ={lq:.2f} -> LSS={harmonic(lfs, orl, lq):.3f}  (plain avg {(lfs+orl+lq)/3:.3f})")

no steering (wrong language, good answer)      LFS=0.00 OR=0.95 LQ=0.95 -> LSS=0.000  (plain avg 0.633)
over-steered (right language, gibberish)       LFS=1.00 OR=0.05 LQ=0.05 -> LSS=0.073  (plain avg 0.367)
right language, relevant, but broken grammar   LFS=0.95 OR=0.85 LQ=0.30 -> LSS=0.539  (plain avg 0.700)
balanced                                       LFS=0.80 OR=0.80 LQ=0.80 -> LSS=0.800  (plain avg 0.800)


## Step 9 - Dry-run the official evaluator on fake outputs

Before we have a real system, let's create two fake "systems" in the **exact submission format** and score them with `evaluate.py --skip-judge` (LFS only; the judge needs a GPU):

1. **echo_source**: the "answer" is just the source-language question, i.e. what no steering looks like. We expect LFS ≈ 0.
2. **echo_target**: the "answer" is the *target-language version of the question*. Perfect language, but it isn't an answer, so LFS ≈ 1 and the judge would give a low OR.

This checks that the pipeline works on your machine and shows why LFS alone isn't enough.

In [22]:
import subprocess
from clas.io_utils import write_json

demo_pairs = [("ar", "de"), ("en", "ja"), ("zh", "fr"), ("hi", "ru")]

for system in ["echo_source", "echo_target"]:
    out_dir = Path(f"generations/_sanity/{system}")
    for src, tgt in demo_pairs:
        outs = questions[src] if system == "echo_source" else questions[tgt]
        write_json(out_dir / f"{src}_to_{tgt}.json", {
            "source_language": src, "target_language": tgt,
            "model": "none", "method": system, "track": "t1",
            "results": [{"question_idx": i, "input": q, "output": o, "original_prompt": q}
                        for i, (q, o) in enumerate(zip(questions[src], outs))],
        })
    print(f"\n------ {system} ------")
    r = subprocess.run([sys.executable, "evaluate.py", "--gen-dir", str(out_dir), "--skip-judge"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("evaluate.py failed:\n", r.stderr[-2000:])
    else:
        print((out_dir / "scores.md").read_text())


------ echo_source ------
# CLaS baseline results

Pairs: 4

| scope | LFS | OR | LQ | LSS |
|---|---|---|---|---|
| **overall** | 0.000 | 0.000 | 0.000 | **0.000** |
| → German (de) | 0.000 | 0.000 | 0.000 | 0.000 |
| → French (fr) | 0.000 | 0.000 | 0.000 | 0.000 |
| → Japanese (ja) | 0.000 | 0.000 | 0.000 | 0.000 |
| → Russian (ru) | 0.000 | 0.000 | 0.000 | 0.000 |


------ echo_target ------
# CLaS baseline results

Pairs: 4

| scope | LFS | OR | LQ | LSS |
|---|---|---|---|---|
| **overall** | 0.986 | 0.000 | 0.000 | **0.000** |
| → German (de) | 1.000 | 0.000 | 0.000 | 0.000 |
| → French (fr) | 0.986 | 0.000 | 0.000 | 0.000 |
| → Japanese (ja) | 0.957 | 0.000 | 0.000 | 0.000 |
| → Russian (ru) | 1.000 | 0.000 | 0.000 | 0.000 |



**Reading the output:** OR, LQ and LSS show 0 because we skipped the judge. That's expected. Look at the **LFS** column: ~0 for echo_source, ~1 for echo_target.

## What we should understand
1. **Input/output:** a question in language S goes in, an answer in language T comes out, with no instruction in the prompt.
2. **Data:** FLORES (parallel sentences) is for *finding* language directions. CLaS-Bench (70 parallel questions) is for *measuring*.
3. **Score**: LSS = harmonic mean of language-ID success, judge-rated relevance and judge-rated fluency, so we need all three.
4. **Format:** one `src_to_tgt.json` per pair, which our steering code must also produce.